In [ ]:
import numpy as np
import glob
import re
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size" : 15,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "mathtext.fontset": "stix",
})



In [ ]:
def read_col_probe(filePath, colIdx, skipHeader=1):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx)
    f.close()
    return vec

def read_col_csv(filePath, colIdx, skipHeader):
    with open(filePath, 'r') as f:
        vec = np.genfromtxt(f, skip_header=skipHeader, usecols=colIdx, delimiter=',')
    f.close()
    return vec

In [ ]:

folderVec = [
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_cavity/Re100_q9_srt/probes",
    # "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_cavity/Re400_q9_srt/probes",
    "/Users/maggie/repo/LBM-Program/amrlbm/bin/a_cavity/Re1000_q9_srt/probes",
]

# Re = 100
# Re = 400
Re = 1000

labelVec = [
    "Present LBM D2Q9 BGK",
]

colorVec = [
    # "#05A361",
    "#D33737",
    "#001AFF",
    "#A17808",
    '#FF5733',
    "#119100",
    "#B700FF",
    "#FA7BA1",
]

markerVec = [
    'o',
    '^',
    's',
    "D",
    '*',
]


colIdx = 3
coordStr = 'y'


ghiaYUPath = f"../data/2d/ghia-{Re}-y-u.dat"
ghiaVXPath = f"../data/2d/ghia-{Re}-v-x.dat"

U0 = 0.03


In [ ]:



coordMap = {
    'x': 0,
    'y': 1,
    'z': 2
}
coordIdx = coordMap[coordStr]
if coordStr == "x":
    velStr = "X_V"
elif coordStr == "y":
    velStr = "Y_U"
folderFullVec = [f"{folder}/probe_{velStr}" for folder in folderVec]
coord_pattern = re.compile(r"\((.*?)\)")
coordVecVec = []
dataVecVec = []
for case in range(0, len(folderVec)):
    fileList = sorted(glob.glob(f"{folderFullVec[case]}/probe*.txt"))
    coordVec = []
    dataVec = []
    for file_path in fileList:
        with open(file_path, 'r') as f:
            lines = f.readlines()
            if not lines:
                continue

            first_line = lines[0].strip()
            match = coord_pattern.search(first_line)
            if match:
                coords = match.group(1).split(',')
                coordVec.append(float(coords[coordIdx]))
            
            last_line = lines[-1].strip()
            columns = last_line.split()
            if len(columns) >= (colIdx + 1):
                dataVec.append(float(columns[colIdx])/U0)
    coordVecVec.append(coordVec)
    dataVecVec.append(dataVec)

ghiaCoordVec = []
ghiaDataVec = []

if coordStr == 'x':
    ghiaCoordVec = read_col_csv(ghiaVXPath, 0, 1)
    ghiaDataVec = read_col_csv(ghiaVXPath, 1, 1)
elif coordStr == 'y':
    ghiaCoordVec = read_col_csv(ghiaYUPath, 1, 1)
    ghiaDataVec = read_col_csv(ghiaYUPath, 0, 1)

fig, ax = plt.subplots(1, 1, figsize=(6, 5), facecolor='w', edgecolor='w')
if coordStr == 'x':
    for case in range(0, len(folderVec)):
        ax.plot(coordVecVec[case], dataVecVec[case], marker=markerVec[case], markevery=3, ms=8, fillstyle='none', linestyle='-', color=colorVec[case], label=f'{labelVec[case]} (Re={Re})')
        # ax.plot(coordVecVec[case], dataVecVec[case], linestyle='-', color=colorVec[case], label=f'{labelVec[case]} (Re={Re})')
    ax.plot(ghiaCoordVec, ghiaDataVec, marker='s', fillstyle='full', mew=1.5, c='k', lw=0, label=f'Ghia et al. (Re={Re})')
    ax.set_xlabel(r'$x$')
    ax.set_ylabel(r'$V/U_0$')
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.6, 0.6)
    ax.legend(loc='upper right', frameon=False, fontsize=13)
elif coordStr == 'y':
    for case in range(0, len(folderVec)):
        ax.plot(dataVecVec[case], coordVecVec[case], marker=markerVec[case], markevery=3, ms=8, fillstyle='none', linestyle='-', color=colorVec[case], label=f'{labelVec[case]} (Re={Re})')
        # ax.plot(dataVecVec[case], coordVecVec[case], linestyle='-', color=colorVec[case], label=f'{labelVec[case]} (Re={Re})')
    ax.plot(ghiaDataVec, ghiaCoordVec, marker='s', fillstyle='full', mew=1.5, c='k', lw=0, label=f'Ghia et al. (Re={Re})')
    ax.set_xlabel(r'$U/U_0$')
    ax.set_ylabel(r'$y$')
    ax.set_xlim(-0.6, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right', frameon=False, fontsize=12)

